In [22]:
!pip install qdrant-client -q
!pip install fastembed-gpu -q

In [23]:
import json
import collections
import pandas as pd
from typing import List

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

### Load documents and ground truth data

In [24]:
with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [25]:
gt_df =  pd.read_csv('ground_truth.csv')

### Qdrant client and collections


In [26]:
client = QdrantClient(":memory:")

In [27]:
def collection_exists(collection_name: str) -> bool:

    try:
        existing_collections = [col.name for col in client.get_collections().collections]
        return collection_name in existing_collections

    except:
        return False

In [28]:
def build_collection(name: str, vector_config: dict = None, sparse_vector_config: dict = None):

    try:
        vector_config = vector_config or {}
        sparse_vector_config = sparse_vector_config or {}

        exists = collection_exists(name)

        if exists:
            print(f"Collection '{name}' already exists.")
            return

        client.create_collection(
            collection_name = name,
            vectors_config = vector_config,
            sparse_vectors_config = sparse_vector_config
        )

        print(f"Qdrant collection '{name}' created.")

    except Exception as e:
        print(f"Failed to create collection '{name}': {e}")

### Populating Qdrant collection

In [29]:
def populate_collection(name: str, models_names: dict, documents: List[dict]):

    try:
        exists = collection_exists(name)

        if not exists:
            print(f"Collection '{name}' does not exists.")
            return

        points = []

        for record in documents:

            text_embed = f"{record['term']}: {record['definition']} {record['extra']}"

            dict_vector = {}

            for vector_name, model_name in models_names.items():
                dict_vector[vector_name] = models.Document(
                    text = text_embed,
                    model = model_name
                )

            point = models.PointStruct(
                id = record['id'],
                vector = dict_vector,
                payload = {
                    'term': record['term'],
                    'description': f"{record['definition']} {record['extra']}",
                    'models_used': models_names
                }
            )

            points.append(point)

        client.upsert(
            collection_name = name,
            points=points
        )

        print(f"Successfully populated collection '{name}' with {len(points)} records.")

    except Exception as e:
        print(f"An error occurred: {e}")

### Semantic search

In [30]:
def semantic_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    vector_config = collection.config.params.vectors

    if isinstance(vector_config, dict) and len(vector_config) > 1:
        print(f"Multiple dense embeddings used in collection '{collection_name}'. Cannot perform semantic search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    if len(used_models) != 1:
        print(f"Collection '{collection_name}' must have exactly one model for semantic search.")
        return

    vector_name, model_name = next(iter(used_models.items()))
    query_doc = models.Document(text=question, model=model_name)

    results = client.query_points(
        collection_name=collection_name,
        query=query_doc,
        using=vector_name,
        limit=limit,
        with_payload=True
    )

    ids = [res.id for res in results.points]
    return ids

### Keyword search

In [31]:
def keyword_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    sparse_vector_config = collection.config.params.sparse_vectors

    if isinstance(sparse_vector_config, dict) and len(sparse_vector_config) > 1:
        print(f"Multiple sparse embeddings used in collection '{collection_name}'. Cannot perform keyword search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    if len(used_models) != 1:
        print(f"Collection '{collection_name}' must have exactly one model for keyword search.")
        return

    vector_name, model_name = next(iter(used_models.items()))
    query_doc = models.Document(text=question, model=model_name)

    results = client.query_points(
        collection_name=collection_name,
        query=query_doc,
        using=vector_name,
        limit=limit,
        with_payload=True
    )

    ids = [res.id for res in results.points]
    return ids


### Multi stage search

In [32]:
def multi_stage_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    vector_config = collection.config.params.vectors
    sparse_vector_config = collection.config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 1:
        print("Cannot perform multi-stage search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():
        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:
        print("Both dense and sparse vectors are required for multi-stage search.")
        return

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=sparse["model"]
        ),
        using=sparse["name"],
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=dense["model"]
                ),
                using=dense["name"],
                limit=2 * limit
            )
        ],
        limit=limit,
        with_payload=True
    )

    ids = [res.id for res in results.points]
    return ids

### Re-ranking fusion search

In [33]:
def rrf_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    vector_config = collection.config.params.vectors
    sparse_vector_config = collection.config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 2:
        print("Both dense and sparse vectors are required for RRF search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():
        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:
        print("Dense and sparse vector fields not found in the collection.")
        return

    results = client.query_points(
        collection_name=collection_name,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=dense["model"]
                ),
                using=dense["name"],
                limit=2 * limit
            ),
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=sparse["model"]
                ),
                using=sparse["name"],
                limit=2 * limit
            )
        ],
        limit=limit,
        with_payload=True
    )

    ids = [res.id for res in results.points]
    return ids

In [34]:
build_collection(
    name = 'keyword-search-collection',
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='keyword-search-collection',
    models_names={
        'sparse_text': 'Qdrant/bm25'
    },
    documents=documents
)

Qdrant collection 'keyword-search-collection' created.
Successfully populated collection 'keyword-search-collection' with 829 records.


In [35]:
build_collection(
    name = 'semantic-search-collection',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    }
)
populate_collection(
    name='semantic-search-collection',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
    },
    documents=documents
)

Qdrant collection 'semantic-search-collection' created.
Successfully populated collection 'semantic-search-collection' with 829 records.


In [36]:
build_collection(
    name = 'hybrid-search-collection',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    },
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='hybrid-search-collection',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
        'sparse_text': 'Qdrant/bm25'
    },
    documents=documents
)

Qdrant collection 'hybrid-search-collection' created.
Successfully populated collection 'hybrid-search-collection' with 829 records.


In [37]:
def score_hit_rate(search_results: dict) -> float:

    if not search_results:
        return 0.0

    hits = 0
    for query_id, docs in search_results.items():
        if query_id in docs:
            hits += 1

    hit_rate = hits / len(search_results)
    return hit_rate

In [38]:
def score_mrr(search_results: dict) -> float:

    if not search_results:
        return 0.0

    mrr = 0.0

    for query_id, docs in search_results.items():
        reciprocal_rank = 0.0

        for index, doc_id in enumerate(docs):
            if doc_id == query_id:
                reciprocal_rank = 1.0 / (index + 1)
                break

        mrr += reciprocal_rank

    mrr = mrr / len(search_results)
    return mrr

### Results

In [39]:
search_functions = {
    "keyword":  lambda q: keyword_search(q, collection_name="keyword-search-collection"),
    "semantic": lambda q: semantic_search(q, collection_name="semantic-search-collection"),
    "multi":    lambda q: multi_stage_search(q, collection_name="hybrid-search-collection"),
    "rrf":      lambda q: rrf_search(q, collection_name="hybrid-search-collection"),
}

In [40]:
keyword_search_results = {}
semantic_search_results = {}
multi_stage_search_results = {}
rrf_search_results = {}

for i in range(len(gt_df)):

    question = gt_df.iloc[i].question
    question_id = gt_df.iloc[i].id

    keyword_search_results[question_id] = search_functions["keyword"](question) or []
    semantic_search_results[question_id] = search_functions["semantic"](question) or []
    multi_stage_search_results[question_id] = search_functions["multi"](question) or []
    rrf_search_results[question_id] = search_functions["rrf"](question) or []

### Evaluation

In [41]:
all_results = {
    "Keyword Search": keyword_search_results,
    "Semantic Search": semantic_search_results,
    "Multi-Stage Search": multi_stage_search_results,
    "RRF Search": rrf_search_results,
}

print(f"{'Method':<20} {'MRR':<10} {'HitRate':<10}")
print("-" * 40)

for method, results in all_results.items():
    mrr = score_mrr(results)
    hit_rate = score_hit_rate(results)
    print(f"{method:<20} {mrr:<10.4f} {hit_rate:<10.4f}")

Method               MRR        HitRate   
----------------------------------------
Keyword Search       0.7162     0.8347    
Semantic Search      0.8449     0.9288    
Multi-Stage Search   0.8013     0.9397    
RRF Search           0.8638     0.9421    


### **Selected Search Method**

Based on our evaluation, **RRF Search** consistently achieves the highest MRR (0.8638) and HitRate (0.9421), outperforming both Semantic and Multi-Stage approaches. Therefore, we will adopt **RRF Search** as our primary retrieval method for this project.
